In [1]:

# =========================================================
# PRUEBA DE WILCOXON SOBRE ROC-AUC POR MODELO
# =========================================================

from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import wilcoxon


# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
# Este notebook se ejecuta desde la carpeta:
# 3_notebooks/

RUTA_BASE = Path("..")

ARCHIVOS_MODELOS = {
    "Árbol de Decisión": RUTA_BASE / "4_results" / "modelo_dt" / "dt_detalle.csv",
    "K-Nearest Neighbors": RUTA_BASE / "4_results" / "modelo_knn" / "knn_detalle.csv",
    "Regresión Logística": RUTA_BASE / "4_results" / "modelo_lr" / "lr_detalle.csv",
    "Random Forest": RUTA_BASE / "4_results" / "modelo_rf" / "rf_detalle.csv",
}

CARPETA_SALIDA = RUTA_BASE / "4_results" / "wilcoxon"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

RUTA_RESULTADOS = CARPETA_SALIDA / "wilcoxon_roc_auc_modelos.csv"


# =========================================================
# PARÁMETROS DE LA PRUEBA
# =========================================================
VALOR_AZAR = 0.5
COLUMNA_AUC = "cv_roc_auc"
ALTERNATIVA = "greater"  # H1: mediana ROC-AUC > 0.5


# =========================================================
# FUNCIÓN PARA APLICAR WILCOXON
# =========================================================
def analizar_modelo(nombre_modelo, ruta_archivo):
    """
    Lee el archivo detalle de un modelo y aplica la prueba de Wilcoxon
    sobre los valores de ROC-AUC contra el valor 0.5.
    """

    if not ruta_archivo.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta_archivo}")

    df = pd.read_csv(ruta_archivo)

    if COLUMNA_AUC not in df.columns:
        raise ValueError(
            f"El archivo {ruta_archivo} no contiene la columna '{COLUMNA_AUC}'. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    auc = df[COLUMNA_AUC].dropna()

    # Diferencia respecto al rendimiento esperado al azar
    diferencias = auc - VALOR_AZAR

    # Wilcoxon no funciona si todas las diferencias son exactamente cero
    if np.allclose(diferencias, 0):
        estadistico = np.nan
        p_value = np.nan
    else:
        estadistico, p_value = wilcoxon(
            diferencias,
            alternative=ALTERNATIVA,
            zero_method="wilcox"
        )

    resultado = {
        "modelo": nombre_modelo,
        "n_observaciones": len(auc),
        "roc_auc_media": auc.mean(),
        "roc_auc_mediana": auc.median(),
        "roc_auc_std": auc.std(),
        "roc_auc_min": auc.min(),
        "roc_auc_max": auc.max(),
        "valor_referencia_azar": VALOR_AZAR,
        "wilcoxon_statistic": estadistico,
        "p_value": p_value,
        "alternative": "ROC-AUC > 0.5",
    }

    return resultado


# =========================================================
# EJECUCIÓN PARA TODOS LOS MODELOS
# =========================================================
resultados = []

for nombre_modelo, ruta_archivo in ARCHIVOS_MODELOS.items():
    print(f"Procesando: {nombre_modelo}")
    resultado = analizar_modelo(nombre_modelo, ruta_archivo)
    resultados.append(resultado)

df_resultados = pd.DataFrame(resultados)


# =========================================================
# INTERPRETACIÓN AUTOMÁTICA
# =========================================================
def interpretar_fila(row, alpha=0.05):
    """
    Genera una interpretación breve considerando significancia estadística
    y cercanía práctica del ROC-AUC al azar.
    """

    if pd.isna(row["p_value"]):
        return "No evaluable"

    if row["p_value"] < alpha and row["roc_auc_mediana"] > 0.5:
        if row["roc_auc_mediana"] < 0.55:
            return "Diferencia estadísticamente significativa, pero rendimiento cercano al azar"
        else:
            return "Rendimiento estadísticamente superior al azar"
    else:
        return "No se observa rendimiento significativamente superior al azar"


df_resultados["interpretacion"] = df_resultados.apply(interpretar_fila, axis=1)


# =========================================================
# REDONDEO PARA TABLA FINAL
# =========================================================
columnas_redondear = [
    "roc_auc_media",
    "roc_auc_mediana",
    "roc_auc_std",
    "roc_auc_min",
    "roc_auc_max",
    "wilcoxon_statistic",
    "p_value",
]

df_resultados_redondeado = df_resultados.copy()

for col in columnas_redondear:
    df_resultados_redondeado[col] = df_resultados_redondeado[col].round(6)


# =========================================================
# GUARDAR RESULTADOS
# =========================================================
df_resultados_redondeado.to_csv(RUTA_RESULTADOS, index=False, encoding="utf-8-sig")

print("\nResultados guardados en:")
print(RUTA_RESULTADOS)

df_resultados_redondeado


Procesando: Árbol de Decisión
Procesando: K-Nearest Neighbors
Procesando: Regresión Logística
Procesando: Random Forest

Resultados guardados en:
..\4_results\wilcoxon\wilcoxon_roc_auc_modelos.csv


,modelo,n_observaciones,roc_auc_media,roc_auc_mediana,roc_auc_std,roc_auc_min,roc_auc_max,valor_referencia_azar,wilcoxon_statistic,p_value,alternative,interpretacion
0,Árbol de Decisión,6000,0.500661,0.500,0.058118,0.320,0.720,0.5,8097255.0,0.602752,ROC-AUC > 0.5,No se observa rendimiento significativamente s...
1,K-Nearest Neighbors,6000,0.500513,0.500,0.057727,0.289,0.745,0.5,8965781.0,0.294184,ROC-AUC > 0.5,No se observa rendimiento significativamente s...
2,Regresión Logística,6000,0.500483,0.500,0.076269,0.256,0.778,0.5,8937529.0,0.354285,ROC-AUC > 0.5,No se observa rendimiento significativamente s...
3,Random Forest,6000,0.499795,0.501,0.079293,0.216,0.753,0.5,8964851.0,0.378672,ROC-AUC > 0.5,No se observa rendimiento significativamente s...
